# 2.4 Vocab, StringStore, and Lexemes


## 💾 Deep Dive into Memory Efficiency

In Module 1, we briefly touched on spaCy's memory management. Let's look closer.

NLP involves handling massive arrays of text. If you store the string "the" a million times, you waste memory. spaCy stores strings exactly once in the `StringStore`, converting them to 64-bit integer hashes.


In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")

# Adding and retrieving from the StringStore
# You can add a new word to the vocabulary
hash_id = nlp.vocab.strings.add("spacy_rocks")
print(f"Hash ID for 'spacy_rocks': {hash_id}")

# You can look it up by string or by hash
print(f"Lookup by string: {nlp.vocab.strings['spacy_rocks']}")
print(f"Lookup by hash: {nlp.vocab.strings[hash_id]}")


Hash ID for 'spacy_rocks': 11057445964824623400
Lookup by string: 11057445964824623400
Lookup by hash: spacy_rocks


## 🧩 Lexemes

Whenever you access the `nlp.vocab` using a string or a hash, spaCy returns a **Lexeme**.

A `Token` is a word *in a specific context* (in a sentence).
A `Lexeme` is a word *in the dictionary* (out of context).

Lexemes don't have POS tags or dependencies (because those depend on the sentence context), but they do have lexical attributes like `is_alpha` or `shape_`.


In [2]:
lexeme = nlp.vocab["Batman"]

print(f"Lexeme text: {lexeme.text}")
print(f"Lexeme hash: {lexeme.orth}")
print(f"Is title case?: {lexeme.is_title}")
print(f"Shape: {lexeme.shape_}")

# Notice this will throw an error if you uncomment it, because a Lexeme has no POS tag!
# print(lexeme.pos_)


Lexeme text: Batman
Lexeme hash: 15780808606389958345
Is title case?: True
Shape: Xxxxx


## ⚠️ The Danger of Unshared Vocabularies

Because of this hash system, two different `nlp` objects do not share the same `StringStore` unless explicitly designed to. 

If you create a `Doc` with `nlp_english` and try to add a Span to it from `nlp_german`, spaCy will crash or give garbage data, because Hash `12345` means "coffee" in English but might mean something completely different (or not exist) in the German vocabulary!

**Golden Rule:** Always ensure your `Doc`, `Token`, and `Span` objects share the exact same `Vocab` object when interacting.
